# Smart Recipe Nutritionist - Development Notebook

This notebook documents the experimental workflow used to build the artifacts consumed by the command-line application.

It covers:

- loading the raw Epicurious-style recipe dataset;
- detecting binary ingredient columns;
- training baseline and machine learning models;
- selecting a final ingredient-combination classifier;
- saving the trained model and feature columns;
- preparing the processed recipe similarity file;
- optionally building a local nutrition facts CSV.

> The raw dataset file `data/epi_r.csv` is optional for normal app usage and is not included in the GitHub repository. It is only needed to reproduce the training/data-preparation workflow.


## 0. Imports and Project Paths

In [1]:
from pathlib import Path
import sys
import os

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

sys.path.insert(0, str(SRC_DIR))

from recipes import RecipesData, NutritionLookup

DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)

PROJECT_ROOT: d:\Git\smart-recipe-nutritionist
DATA_DIR: d:\Git\smart-recipe-nutritionist\data
MODELS_DIR: d:\Git\smart-recipe-nutritionist\models


## 1. Load Raw Recipe Dataset

The raw file `epi_r.csv` is used for experimentation and artifact generation.

It should be placed locally at:

```text
data/epi_r.csv
```

The final CLI app does **not** require this raw file if the processed data and model artifacts are already present.


In [2]:
raw_data_path = DATA_DIR / "epi_r.csv"

if not raw_data_path.exists():
    raise FileNotFoundError(
        "Raw dataset not found. Place epi_r.csv inside data/ if you want to reproduce training. "
        "The CLI app can still run without this file if processed data and model files are present."
    )

recipes_data = RecipesData(data_path=raw_data_path)
df = recipes_data.load()
ingredient_cols = recipes_data.detect_ingredient_columns()

print("Dataset shape:", df.shape)
print("Detected ingredient columns:", len(ingredient_cols))
df[["title", "rating"]].head()

Dataset shape: (20052, 680)
Detected ingredient columns: 673


,title,rating
0,"Lentil, Apple, and Turkey Wrap",2.500
1,Boudin Blanc Terrine with Red Onion Confit,4.375
2,Potato and Fennel Soup Hodge,3.750
3,Mahi-Mahi in Tomato Olive Sauce,5.000
4,Spinach Noodle Casserole,3.125


## 2. Prepare Ingredient Matrix and Targets

In [3]:
X = recipes_data.ingredient_matrix()
y_reg = df["rating"].copy()

mask_reg = y_reg.notna()
X_reg = X.loc[mask_reg].copy()
y_reg = y_reg.loc[mask_reg].copy()

y_cls_int = y_reg.round().clip(0, 5).astype(int)

def to_label(value: int) -> str:
    if value in (0, 1):
        return "bad"
    if value in (2, 3):
        return "so-so"
    return "great"

y_cls3 = y_cls_int.map(to_label)

print("X_reg:", X_reg.shape)
print("y_reg:", y_reg.shape)
print("y_cls_int:", y_cls_int.shape)
print("y_cls3:", y_cls3.shape)
y_cls3.value_counts()

X_reg: (20052, 673)
y_reg: (20052,)
y_cls_int: (20052,)
y_cls3: (20052,)


rating
great    15907
so-so     2145
bad       2000
Name: count, dtype: int64

## 3. Train/Test Splits

In [4]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

X_train_ci, X_test_ci, y_train_ci, y_test_ci = train_test_split(
    X_reg,
    y_cls_int,
    test_size=0.2,
    random_state=42,
    stratify=y_cls_int
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_reg,
    y_cls3,
    test_size=0.2,
    random_state=42,
    stratify=y_cls3
)

print("Regression split:", X_train_r.shape, X_test_r.shape)
print("Integer classification split:", X_train_ci.shape, X_test_ci.shape)
print("3-class classification split:", X_train_c.shape, X_test_c.shape)

Regression split: (16041, 673) (4011, 673)
Integer classification split: (16041, 673) (4011, 673)
3-class classification split: (16041, 673) (4011, 673)


## 4. Regression Baseline

In [5]:
naive_reg = DummyRegressor(strategy="mean")
naive_reg.fit(X_train_r, y_train_r)
naive_pred = naive_reg.predict(X_test_r)

naive_rmse = mean_squared_error(y_test_r, naive_pred) ** 0.5
print("Naive RMSE:", naive_rmse)

Naive RMSE: 1.3052174279518536


## 5. Regression Models

In [6]:
reg_models = {
    "rf_reg": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100],
            "max_depth": [20],
            "min_samples_leaf": [1, 2],
        }
    ),
    "gbr_reg": (
        GradientBoostingRegressor(random_state=42),
        {
            "n_estimators": [100],
            "learning_rate": [0.1],
            "max_depth": [2, 3],
        }
    ),
    "dt_reg": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [10, 20],
            "min_samples_leaf": [1, 2],
        }
    ),
}

reg_results = []
best_reg = None
best_reg_name = None
best_reg_cv = float("inf")

for name, (model, params) in reg_models.items():
    gs = GridSearchCV(
        model,
        params,
        scoring="neg_mean_squared_error",
        cv=2,
        n_jobs=-1,
        error_score="raise"
    )
    gs.fit(X_train_r, y_train_r)

    test_pred = gs.best_estimator_.predict(X_test_r)
    test_rmse = mean_squared_error(y_test_r, test_pred) ** 0.5
    cv_rmse = (-gs.best_score_) ** 0.5

    reg_results.append({
        "model": name,
        "best_params": gs.best_params_,
        "cv_rmse": cv_rmse,
        "test_rmse": test_rmse,
    })

    if cv_rmse < best_reg_cv:
        best_reg_cv = cv_rmse
        best_reg = gs.best_estimator_
        best_reg_name = name

pd.DataFrame(reg_results).sort_values("test_rmse")

,model,best_params,cv_rmse,test_rmse
0,rf_reg,"{'max_depth': 20, 'min_samples_leaf': 2, 'n_es...",1.228000,1.206727
1,gbr_reg,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",1.236718,1.223254
2,dt_reg,"{'max_depth': 10, 'min_samples_leaf': 2}",1.282848,1.275154


## 6. Classification Baselines

In [7]:
naive_cls_int = DummyClassifier(strategy="most_frequent")
naive_cls_int.fit(X_train_ci, y_train_ci)
naive_pred_int = naive_cls_int.predict(X_test_ci)

print("Naive integer accuracy:", accuracy_score(y_test_ci, naive_pred_int))

naive_cls = DummyClassifier(strategy="most_frequent")
naive_cls.fit(X_train_c, y_train_c)
naive_pred = naive_cls.predict(X_test_c)

print("Naive 3-class accuracy:", accuracy_score(y_test_c, naive_pred))
print("Naive 3-class weighted F1:", f1_score(y_test_c, naive_pred, average="weighted"))

Naive integer accuracy: 0.6576913487908252
Naive 3-class accuracy: 0.7933183744702069
Naive 3-class weighted F1: 0.7018876873527592


## 7. Integer Classification Models

In [8]:
cls_int_models = {
    "rf_cls_int": (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100],
            "max_depth": [20],
            "class_weight": [None, "balanced"],
        }
    ),
    "logreg_int": (
        LogisticRegression(max_iter=1000),
        {
            "C": [1.0],
            "class_weight": [None, "balanced"],
        }
    ),
    "extra_trees_int": (
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100],
            "max_depth": [20],
            "class_weight": [None, "balanced"],
        }
    ),
    "dt_cls_int": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [10, 20],
            "class_weight": [None, "balanced"],
        }
    ),
}

cls_int_results = []
best_cls_int = None
best_cls_int_name = None
best_cls_int_cv = float("-inf")

for name, (model, params) in cls_int_models.items():
    gs = GridSearchCV(
        model,
        params,
        scoring="accuracy",
        cv=2,
        n_jobs=-1,
        error_score="raise"
    )
    gs.fit(X_train_ci, y_train_ci)

    pred = gs.best_estimator_.predict(X_test_ci)
    test_acc = accuracy_score(y_test_ci, pred)

    cls_int_results.append({
        "model": name,
        "best_params": gs.best_params_,
        "cv_accuracy": gs.best_score_,
        "test_accuracy": test_acc,
    })

    if gs.best_score_ > best_cls_int_cv:
        best_cls_int_cv = gs.best_score_
        best_cls_int = gs.best_estimator_
        best_cls_int_name = name

pd.DataFrame(cls_int_results).sort_values("test_accuracy", ascending=False)

,model,best_params,cv_accuracy,test_accuracy
2,extra_trees_int,"{'class_weight': None, 'max_depth': 20, 'n_est...",0.676392,0.677636
0,rf_cls_int,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.675083,0.673897
1,logreg_int,"{'C': 1.0, 'class_weight': None}",0.666791,0.673398
3,dt_cls_int,"{'class_weight': None, 'max_depth': 10}",0.666293,0.669409


## 8. Three-Class Classification Models

In [9]:
cls_models_f1 = {
    "rf_cls": (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100],
            "max_depth": [20],
            "class_weight": [None, "balanced"],
        }
    ),
    "logreg": (
        LogisticRegression(max_iter=1000),
        {
            "C": [1.0],
            "class_weight": [None, "balanced"],
        }
    ),
    "extra_trees": (
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            "n_estimators": [100],
            "max_depth": [20],
            "class_weight": [None, "balanced"],
        }
    ),
    "dt_cls": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [10, 20],
            "class_weight": [None, "balanced"],
        }
    ),
}

cls_results = []
best_cls = None
best_cls_name = None
best_cls_cv = float("-inf")

for name, (model, params) in cls_models_f1.items():
    gs = GridSearchCV(
        model,
        params,
        scoring="f1_weighted",
        cv=2,
        n_jobs=-1,
        error_score="raise"
    )
    gs.fit(X_train_c, y_train_c)

    pred = gs.best_estimator_.predict(X_test_c)
    test_acc = accuracy_score(y_test_c, pred)
    test_f1 = f1_score(y_test_c, pred, average="weighted")

    cls_results.append({
        "model": name,
        "best_params": gs.best_params_,
        "cv_weighted_f1": gs.best_score_,
        "test_accuracy": test_acc,
        "test_weighted_f1": test_f1,
    })

    if gs.best_score_ > best_cls_cv:
        best_cls_cv = gs.best_score_
        best_cls = gs.best_estimator_
        best_cls_name = name

pd.DataFrame(cls_results).sort_values("test_weighted_f1", ascending=False)

,model,best_params,cv_weighted_f1,test_accuracy,test_weighted_f1
0,rf_cls,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.757431,0.772625,0.763722
1,logreg,"{'C': 1.0, 'class_weight': None}",0.740768,0.804288,0.743656
2,extra_trees,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.742418,0.731488,0.741029
3,dt_cls,"{'class_weight': None, 'max_depth': 10}",0.739320,0.804288,0.735203


## 9. Optional Voting Ensemble

In [10]:
rf_for_vote = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

logreg_for_vote = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=1000,
)

extra_for_vote = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

vote = VotingClassifier(
    estimators=[
        ("rf", rf_for_vote),
        ("logreg", logreg_for_vote),
        ("extra", extra_for_vote),
    ],
    voting="soft",
)

vote.fit(X_train_c, y_train_c)
vote_pred = vote.predict(X_test_c)

vote_acc = accuracy_score(y_test_c, vote_pred)
vote_f1 = f1_score(y_test_c, vote_pred, average="weighted")

print("Voting accuracy:", vote_acc)
print("Voting weighted F1:", vote_f1)

Voting accuracy: 0.6719022687609075
Voting weighted F1: 0.7026402627389641


## 10. Save Final Classifier and Feature Columns

The final app needs two model artifacts:

```text
models/best_rating_label_model.joblib
models/feature_columns.joblib
```


In [11]:
best_single_f1 = pd.DataFrame(cls_results)["test_weighted_f1"].max()

final_model = vote if vote_f1 > best_single_f1 else best_cls
final_model_name = "voting_classifier" if vote_f1 > best_single_f1 else best_cls_name

print("best_reg_name:", best_reg_name)
print("best_cls_int_name:", best_cls_int_name)
print("best_cls_name:", best_cls_name)
print("best_single_f1:", best_single_f1)
print("vote_f1:", vote_f1)
print("final_model_name:", final_model_name)
print("final_model:", final_model)

assert final_model is not None, "final_model is None"

model_path = MODELS_DIR / "best_rating_label_model.joblib"
feature_columns_path = MODELS_DIR / "feature_columns.joblib"

joblib.dump(final_model, model_path)
joblib.dump(list(X_reg.columns), feature_columns_path)

print("Saved model:", model_path)
print("Saved feature columns:", feature_columns_path)

best_reg_name: rf_reg
best_cls_int_name: extra_trees_int
best_cls_name: rf_cls
best_single_f1: 0.7637222589609511
vote_f1: 0.7026402627389641
final_model_name: rf_cls
final_model: RandomForestClassifier(class_weight='balanced', max_depth=20, n_jobs=-1,
                       random_state=42)
Saved model: d:\Git\smart-recipe-nutritionist\models\best_rating_label_model.joblib
Saved feature columns: d:\Git\smart-recipe-nutritionist\models\feature_columns.joblib


## 11. Decision

For the final CLI prototype, the project uses a 3-class classification model (`bad`, `so-so`, `great`) instead of a regression model.

Reasons:

- the CLI output is more understandable as a label;
- labels are aligned with the user-facing forecast message;
- weighted F1 is more useful than plain accuracy for imbalanced classes.


## 12. Build Processed Similar Recipes CSV

The CLI app uses:

```text
data/similar_recipes.csv
```

This file contains recipe metadata and binary ingredient columns for cosine-similarity search.


In [12]:
keep_cols = ["title", "rating"]

if "url" in df.columns:
    keep_cols.append("url")

similar_df = pd.concat([df[keep_cols], X], axis=1)

if "url" in similar_df.columns:
    similar_df = similar_df[similar_df["url"].notna()]

similar_recipes_path = DATA_DIR / "similar_recipes.csv"
similar_df.to_csv(similar_recipes_path, index=False)

print("Similar recipes CSV created:", similar_recipes_path)
similar_df.head()

Similar recipes CSV created: d:\Git\smart-recipe-nutritionist\data\similar_recipes.csv


,title,rating,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,advance prep required,alabama,alaska,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
0,"Lentil, Apple, and Turkey Wrap",2.500,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,Boudin Blanc Terrine with Red Onion Confit,4.375,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Potato and Fennel Soup Hodge,3.750,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Mahi-Mahi in Tomato Olive Sauce,5.000,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Spinach Noodle Casserole,3.125,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 13. Optional: Build Local Nutrition CSV

The CLI can use:

```text
data/ingredient_nutrition_daily_values.csv
```

This file allows the app to show nutrition facts without relying fully on external API calls.

This step can require a USDA FoodData Central API key. Set it as an environment variable:

Windows PowerShell:

```powershell
$env:USDA_API_KEY="your_api_key_here"
```

macOS/Linux:

```bash
export USDA_API_KEY="your_api_key_here"
```

To avoid accidental API calls, the build flag below is disabled by default.


In [13]:
RUN_USDA_API_BUILD = False

nutrition_csv_path = DATA_DIR / "ingredient_nutrition_daily_values.csv"

if RUN_USDA_API_BUILD:
    lookup = NutritionLookup()
    nutrition_df = lookup.build_csv_from_ingredients(ingredient_cols, nutrition_csv_path)
    print("Nutrition CSV created:", nutrition_csv_path)
else:
    if nutrition_csv_path.exists():
        nutrition_df = pd.read_csv(nutrition_csv_path)
        print("Existing nutrition CSV loaded:", nutrition_csv_path)
    else:
        nutrition_df = pd.DataFrame()
        print(
            "Nutrition CSV was not created. Set RUN_USDA_API_BUILD = True "
            "and configure USDA_API_KEY if you want to generate it."
        )

nutrition_df.head()

Existing nutrition CSV loaded: d:\Git\smart-recipe-nutritionist\data\ingredient_nutrition_daily_values.csv


,ingredient,ingredient_norm,total fat,total carbohydrate,calcium,cholesterol,saturated fat,protein,iron
0,#cakeweek,#cakeweek,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,#wasteless,#wasteless,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22-minute meals,22-minute meals,0.00,5.02,0.15,0.00,0.0,0.00,0.00
3,3-ingredient recipes,3-ingredient recipes,21.18,0.00,1.54,29.67,32.0,52.88,13.89
4,30 days of groceries,30 days of groceries,38.46,0.00,1.85,26.00,59.0,28.80,9.11
